# Phase C2 — Comprehensive Behavioural, Relational and ML EDA

This notebook analyses the normalized observable Dataset V1.

Goals:
1. understand every observable table;
2. verify relational and behavioural structure;
3. understand intervention/outcome behaviour;
4. identify useful observable features;
5. identify leakage and support/selection issues;
6. produce evidence for the final C3 feature contract.

No model training is performed here.
No latent sidecar data is used.

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CURRENT = Path.cwd()

if (CURRENT / "data" / "normalized").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data" / "normalized").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise FileNotFoundError(
        "Could not locate data/normalized."
    )

DATA_DIR = PROJECT_ROOT / "data" / "normalized"
REPORT_DIR = PROJECT_ROOT / "reports" / "eda"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TABLES = [
    "customers",
    "subscriptions",
    "payment_attempts",
    "recovery_cases",
    "recovery_events",
    "action_log",
    "case_outcomes",
    "ml_datamart",
]

data = {}

for name in TABLES:
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing normalized table: {path}")
    data[name] = pd.read_csv(path, low_memory=False)

ml = data["ml_datamart"]

print("Loaded normalized tables:")
for name, df in data.items():
    print(f"  {name:20s} {len(df):>10,} x {len(df.columns)}")


Loaded normalized tables:
  customers                20,000 x 5
  subscriptions            22,078 x 12
  payment_attempts        264,936 x 11
  recovery_cases          103,026 x 19
  recovery_events         986,365 x 9
  action_log              157,423 x 20
  case_outcomes           103,026 x 17
  ml_datamart             157,423 x 45


## 1. ML Grain and Repetition

In [2]:

for col in [
    "action_id",
    "case_id",
    "customer_id",
    "subscription_id",
    "logging_policy",
]:
    if col in ml.columns:
        print(f"{col:20s}: {ml[col].nunique(dropna=True):,} unique")

if {"case_id", "decision_seq"}.issubset(ml.columns):
    rows_per_case = ml.groupby("case_id").size()
    print("\nRows per case:")
    display(rows_per_case.describe())


case_id             : 25,775 unique
customer_id         : 10,924 unique
subscription_id     : 11,596 unique
logging_policy      : 3 unique

Rows per case:


count    25775.000000
mean         6.107585
std          2.739106
min          1.000000
25%          4.000000
50%          5.000000
75%          8.000000
max         13.000000
dtype: float64

## 2. Customers — Behaviour and Distribution

In [3]:

cust = data["customers"]
display(cust.describe(include="all").T)

for col in ["age_band", "state", "locale_pref"]:
    if col in cust.columns:
        display(
            cust[col].value_counts(dropna=False)
            .rename("count").to_frame()
        )


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,20000.0,NaN,NaN,NaN,10000.5,5773.647028,1.0,5000.75,10000.5,15000.25,20000.0
age_band,20000,5,AGE_26_35,6935,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,20000,10,MAHARASHTRA,3009,NaN,NaN,NaN,NaN,NaN,NaN,NaN
locale_pref,20000,3,EN,10914,NaN,NaN,NaN,NaN,NaN,NaN,NaN
signed_up_date,20000,730,2022-04-22 00:00:00,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,count
age_band,
AGE_26_35,6935
AGE_36_45,5050
AGE_18_25,4977
AGE_46_55,2019
AGE_56_PLUS,1019


,count
state,
MAHARASHTRA,3009
UTTAR_PRADESH,2392
KARNATAKA,2353
DELHI,2064
TAMIL_NADU,1925
TELANGANA,1803
ANDHRA_PRADESH,1646
GUJARAT,1623
RAJASTHAN,1605


,count
locale_pref,
EN,10914
HI,5079
HI_EN,4007


## 3. Subscriptions — Commercial and Payment Setup

In [4]:

subs = data["subscriptions"]

for col in [
    "segment", "plan_tier", "cadence",
    "rail", "bank_name", "status"
]:
    if col in subs.columns:
        display(
            subs[col].value_counts(dropna=False)
            .rename("count").to_frame()
        )

if "customer_id" in subs.columns:
    display(
        subs.groupby("customer_id")
        .size()
        .describe()
        .to_frame("subscriptions_per_customer")
    )


,count
segment,
OTT_STREAMING,7730
NEWS_MEDIA,4408
FITNESS,3350
EDUCATION,3329
SAAS_SMB,3261


,count
plan_tier,
BASIC,12297
STANDARD,6533
PREMIUM,3248


,count
cadence,
MONTHLY,22078


,count
rail,
UPI_AUTOPAY,15579
CARD_SI,6499


,count
bank_name,
SBI,5494
HDFC,3337
ICICI,2635
AXIS,2248
PNB,2231
INDUSIND,2180
YES_BANK,2143
KOTAK,1810


,count
status,
ACTIVE,22078


,subscriptions_per_customer
count,20000.000000
mean,1.103900
std,0.305138
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,2.000000


## 4. Payment Attempts — Failure and Retry Behaviour

In [5]:

attempts = data["payment_attempts"]

for col in [
    "outcome",
    "failure_class",
    "is_recovery_attempt",
]:
    if col in attempts.columns:
        display(
            attempts[col].value_counts(dropna=False)
            .rename("count").to_frame()
        )

for col in ["attempt_index", "amount_inr"]:
    if col in attempts.columns:
        display(attempts[col].describe().to_frame())


,count
outcome,
SUCCESS,238138
FAILED,26798


,count
failure_class,
NaN,238138
INSUFFICIENT_FUNDS,13270
BANK_TECHNICAL,5127
MANDATE_PAUSED,3968
MANDATE_CLOSED,3647
CARD_EXPIRED,452
CARD_LOST_STOLEN,334


,count
is_recovery_attempt,
False,264936


,attempt_index
count,264936.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


,amount_inr
count,264936.00000
mean,512.19413
std,696.19636
min,49.00000
25%,149.00000
50%,299.00000
75%,599.00000
max,3999.00000


## 5. Recovery Cases — Case Behaviour

In [6]:

cases = data["recovery_cases"]

for col in [
    "first_failure_class",
    "current_failure_class",
    "value_tier",
    "resolution",
    "policy_id",
]:
    if col in cases.columns:
        display(
            cases[col].value_counts(dropna=False)
            .rename("count").to_frame()
        )

for col in [
    "amount_at_risk",
    "n_decisions",
    "retries_used",
    "contacts_sent",
    "cost_total_inr",
    "fatigue_total",
    "recovered_amount_inr",
    "time_to_recovery_h",
]:
    if col in cases.columns:
        display(cases[col].describe().to_frame())


,count
first_failure_class,
INSUFFICIENT_FUNDS,53353
BANK_TECHNICAL,20623
MANDATE_CLOSED,14588
MANDATE_PAUSED,12814
CARD_EXPIRED,1034
CARD_LOST_STOLEN,614


,count
current_failure_class,
INSUFFICIENT_FUNDS,54314
BANK_TECHNICAL,20019
MANDATE_CLOSED,14588
MANDATE_PAUSED,11176
RATE_LIMIT,1547
CARD_EXPIRED,853
CARD_LOST_STOLEN,529


,count
value_tier,
LOW,62020
MID,30799
HIGH,10207


,count
resolution,
EXPIRED_WINDOW,61089
RECOVERED,34200
STOPPED_POLICY,5162
ESCALATED_UNRESOLVED,1533
STOPPED_OPT_OUT,1042


,count
policy_id,
B0_NONE,26798
B1_PSP_PROXY,25625
B3_RANDOM_LEGAL,25332
B2_DOCTRINE,25271


,n_decisions
count,103026.000000
mean,5.509289
std,5.586530
min,0.000000
25%,2.000000
50%,2.000000
75%,9.000000
max,24.000000


... (2 outputs left out)

,cost_total_inr
count,103026.000000
mean,1.464603
std,6.475767
min,0.000000
25%,0.000000
50%,0.500000
75%,1.150000
max,51.750000


,fatigue_total
count,103026.000000
mean,0.565731
std,0.855803
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,4.000000


,recovered_amount_inr
count,103026.000000
mean,204.846835
std,552.912071
min,0.000000
25%,0.000000
50%,0.000000
75%,149.000000
max,3999.000000


,time_to_recovery_h
count,34200.000000
mean,61.892807
std,72.086112
min,3.000000
25%,18.000000
50%,24.000000
75%,71.000000
max,385.000000


## 6. Recovery Events — Lifecycle

In [7]:

events = data["recovery_events"]

for col in ["event_type", "policy_id"]:
    if col in events.columns:
        display(
            events[col].value_counts(dropna=False)
            .rename("count").to_frame()
        )

if {"case_id", "ts_h"}.issubset(events.columns):
    non_monotonic = (
        events.sort_values(["case_id", "ts_h"])
        .groupby("case_id")["ts_h"]
        .apply(lambda s: not s.is_monotonic_increasing)
        .sum()
    )
    print(
        "Cases with non-monotonic event time:",
        int(non_monotonic)
    )


,count
event_type,
DECISION_MADE,574486
CASE_OPENED,103026
CASE_CLOSED,103026
ACTION_EXECUTED,101514
NOTIFICATION_SENT,55909
PAYMENT_SUCCEEDED,34200
CUSTOMER_RESPONDED,8852
HUMAN_ACTION,1724
ESCALATION_CREATED,1724


,count
policy_id,
B1_PSP_PROXY,380458
B3_RANDOM_LEGAL,296350
B2_DOCTRINE,202365
B0_NONE,107192


Cases with non-monotonic event time: 0


## 7. Action and Policy Behaviour

In [8]:

actions = data["action_log"]

for col in [
    "action_type",
    "channel",
    "policy_id",
    "status",
]:
    if col in actions.columns:
        display(
            actions[col].value_counts(dropna=False)
            .rename("count").to_frame()
        )

if {"policy_id", "action_type"}.issubset(actions.columns):
    print("Policy × action counts:")
    display(
        pd.crosstab(
            actions["policy_id"],
            actions["action_type"]
        )
    )

if "case_id" in actions.columns:
    actions_per_case = actions.groupby("case_id").size()
    display(actions_per_case.describe().to_frame("actions_per_case"))


,count
action_type,
RETRY,101514
SEND_PAYMENT_LINK,30884
SEND_REMINDER,22649
REQUEST_PM_UPDATE,2376


,count
channel,
NaN,101514
SMS,31410
WHATSAPP,12285
EMAIL,12214


,count
policy_id,
B3_RANDOM_LEGAL,64926
B1_PSP_PROXY,49915
B2_DOCTRINE,42582


Policy × action counts:


action_type,REQUEST_PM_UPDATE,RETRY,SEND_PAYMENT_LINK,SEND_REMINDER
policy_id,,,,
B1_PSP_PROXY,0,42791,7124,0
B2_DOCTRINE,259,30849,6314,5160
B3_RANDOM_LEGAL,2117,27874,17446,17489


,actions_per_case
count,25775.000000
mean,6.107585
std,2.739106
min,1.000000
25%,4.000000
50%,5.000000
75%,8.000000
max,13.000000


## 8. Terminal Case Outcomes

In [9]:

outcomes = data["case_outcomes"]

for col in [
    "resolution",
    "recovered_flag",
    "policy_id",
]:
    if col in outcomes.columns:
        display(
            outcomes[col].value_counts(dropna=False)
            .rename("count").to_frame()
        )

for col in [
    "recovered_amount_inr",
    "recovery_rate",
    "attempts_used",
    "contacts_used",
    "cost_total_inr",
    "fatigue_total",
    "time_to_recovery_h",
]:
    if col in outcomes.columns:
        display(outcomes[col].describe().to_frame())


,count
resolution,
EXPIRED_WINDOW,61089
RECOVERED,34200
STOPPED_POLICY,5162
ESCALATED_UNRESOLVED,1533
STOPPED_OPT_OUT,1042


,count
recovered_flag,
False,68826
True,34200


,count
policy_id,
B0_NONE,26798
B1_PSP_PROXY,25625
B3_RANDOM_LEGAL,25332
B2_DOCTRINE,25271


,recovered_amount_inr
count,103026.000000
mean,204.846835
std,552.912071
min,0.000000
25%,0.000000
50%,0.000000
75%,149.000000
max,3999.000000


,recovery_rate
count,103026.000000
mean,0.331955
std,0.470917
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


,cost_total_inr
count,103026.000000
mean,1.464603
std,6.475767
min,0.000000
25%,0.000000
50%,0.500000
75%,1.150000
max,51.750000


,fatigue_total
count,103026.000000
mean,0.565731
std,0.855803
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,4.000000


,time_to_recovery_h
count,34200.000000
mean,61.892807
std,72.086112
min,3.000000
25%,18.000000
50%,24.000000
75%,71.000000
max,385.000000


## 9. ML Datamart — Target and Intervention Behaviour

In [10]:

target = "y" if "y" in ml.columns else "y_intervention"
ml[target] = pd.to_numeric(ml[target], errors="coerce")

print("Target:", target)
print("Target positive rate:", ml[target].mean())

for col in [
    "action_type",
    "rail",
    "failure_class",
    "logging_policy",
]:
    if col in ml.columns:
        print(f"\nObserved outcome by {col}:")
        display(
            ml.groupby(col)[target]
            .agg(["count", "mean"])
            .sort_values("mean", ascending=False)
        )


Target: y
Target positive rate: 0.2281305781239082

Observed outcome by action_type:


,count,mean
action_type,,
RETRY,101514,0.266574
REQUEST_PM_UPDATE,2376,0.242003
SEND_PAYMENT_LINK,30884,0.176499
SEND_REMINDER,22649,0.124774



Observed outcome by rail:


,count,mean
rail,,
CARD_SI,33098,0.284609
UPI_AUTOPAY,124325,0.213095



Observed outcome by logging_policy:


,count,mean
logging_policy,,
B2_DOCTRINE,42582,0.304871
B3_RANDOM_LEGAL,64926,0.205804
B1_PSP_PROXY,49915,0.191706


## 10. Timing and Salary-Proxy Behaviour

In [11]:

if "proxy_days_to_salary" in ml.columns:
    tmp = ml[
        ["proxy_days_to_salary", target]
    ].dropna().copy()

    bins = [-np.inf, 0, 3, 7, 14, 21, np.inf]
    labels = [
        "<=0", "1-3", "4-7",
        "8-14", "15-21", "22+"
    ]

    tmp["salary_bin"] = pd.cut(
        tmp["proxy_days_to_salary"],
        bins=bins,
        labels=labels,
    )

    timing_summary = (
        tmp.groupby("salary_bin", observed=False)[target]
        .agg(["count", "mean"])
    )

    timing_summary["success_rate_pct"] = (
        timing_summary["mean"] * 100
    )

    display(timing_summary)

    plt.figure(figsize=(10, 4))
    plt.bar(
        timing_summary.index.astype(str),
        timing_summary["success_rate_pct"]
    )
    plt.ylabel("Observed success rate (%)")
    plt.xlabel("Salary-proxy bin")
    plt.title("Observed Success by Salary-Proxy Timing")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()


## 11. Candidate Feature Behaviour

In [12]:

candidate_path = (
    PROJECT_ROOT
    / "reports"
    / "ml_contract"
    / "candidate_features.csv"
)

if candidate_path.exists():
    candidates = pd.read_csv(candidate_path)

    for feature in candidates["column"].dropna().unique():

        if feature not in ml.columns:
            continue

        print(f"\n### {feature}")
        print(
            "Missing rate:",
            f"{ml[feature].isna().mean():.2%}"
        )

        if pd.api.types.is_numeric_dtype(
            ml[feature]
        ):
            display(
                ml[feature].describe()
                .to_frame()
            )
        else:
            display(
                ml[feature]
                .value_counts(dropna=False)
                .head(20)
                .rename("count")
                .to_frame()
            )



### rail
Missing rate: 0.00%


,count
rail,
UPI_AUTOPAY,124325
CARD_SI,33098



### bank_name
Missing rate: 0.00%


,count
bank_name,
SBI,40691
HDFC,22618
ICICI,19642
AXIS,16591
PNB,15907
INDUSIND,14489
YES_BANK,14357
KOTAK,13128



### segment
Missing rate: 0.00%


,count
segment,
OTT_STREAMING,40630
SAAS_SMB,39959
EDUCATION,33304
FITNESS,26129
NEWS_MEDIA,17401


... (54 outputs left out)


### action_type
Missing rate: 0.00%


,count
action_type,
RETRY,101514
SEND_PAYMENT_LINK,30884
SEND_REMINDER,22649
REQUEST_PM_UPDATE,2376



### channel
Missing rate: 64.48%


,count
channel,
NaN,101514
SMS,31410
WHATSAPP,12285
EMAIL,12214


## 12. Feature–Target Relationships

This section is exploratory, not causal.

We will examine:
- numeric feature distributions by target
- categorical target rates
- action × state interactions
- failure class × action
- rail × action
- timing × action
- history × outcome

Do not call a high observed action success rate a causal treatment effect.

In [13]:

# Example high-value interaction tables.
for group_cols in [
    ["failure_class", "action_type"],
    ["rail", "action_type"],
    ["logging_policy", "action_type"],
]:
    if all(col in ml.columns for col in group_cols):
        print("\n", " × ".join(group_cols))
        display(
            ml.groupby(group_cols)[target]
            .agg(["size", "mean"])
            .rename(columns={
                "size": "rows",
                "mean": "observed_success_rate"
            })
            .sort_values("observed_success_rate", ascending=False)
        )



 rail × action_type


rows  observed_success_rate
rail        action_type                                    
CARD_SI     RETRY              25196               0.330648
UPI_AUTOPAY REQUEST_PM_UPDATE   1946               0.245632
            RETRY              76318               0.245420
CARD_SI     REQUEST_PM_UPDATE    430               0.225581
UPI_AUTOPAY SEND_PAYMENT_LINK  26913               0.182031
CARD_SI     SEND_PAYMENT_LINK   3971               0.139008
            SEND_REMINDER       3501               0.125678
UPI_AUTOPAY SEND_REMINDER      19148               0.124608


 logging_policy × action_type


rows  observed_success_rate
logging_policy  action_type                                    
B2_DOCTRINE     RETRY              30849               0.345684
B3_RANDOM_LEGAL RETRY              27874               0.302935
B2_DOCTRINE     SEND_PAYMENT_LINK   6314               0.256573
B3_RANDOM_LEGAL REQUEST_PM_UPDATE   2117               0.243269
B2_DOCTRINE     REQUEST_PM_UPDATE    259               0.231660
B1_PSP_PROXY    SEND_PAYMENT_LINK   7124               0.226839
                RETRY              42791               0.185857
B3_RANDOM_LEGAL SEND_PAYMENT_LINK  17446               0.126963
                SEND_REMINDER      17489               0.125107
B2_DOCTRINE     SEND_REMINDER       5160               0.123643

## 13. Leakage / Decision-Time Review

Use the C0 classification and availability reports as the starting point.

Review every candidate feature against the rule:

> the value must be computable using information available before the decision snapshot.

Never use:
- target
- current intervention outcome
- terminal resolution
- recovered amount
- realized post-decision outcome information
- latent sidecar
- counterfactual outcomes.

In [14]:

classification_path = (
    PROJECT_ROOT
    / "reports"
    / "ml_contract"
    / "column_classification.csv"
)

availability_path = (
    PROJECT_ROOT
    / "reports"
    / "ml_contract"
    / "feature_availability.csv"
)

if classification_path.exists():
    display(
        pd.read_csv(classification_path)
    )

if availability_path.exists():
    display(
        pd.read_csv(availability_path)
    )


,column,classification,usable_for_ml,reason,source
0,logging_policy,POLICY / LOGGING,False,Describes the policy that generated/logged the...,ml_datamart
1,case_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
2,subscription_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
3,customer_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
4,decision_seq,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart
5,decision_ts_h,TEMPORAL SOURCE FIELD,False,Used to derive decision-time temporal features...,ml_datamart
6,failure_class_at_decision,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart
7,first_failure_class,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart
8,rail,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart
9,bank_name,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart


,column,available_at_case_open,available_at_decision,known_only_after_execution,post_outcome,usable_for_ml,classification,reason
0,logging_policy,False,True,False,False,False,POLICY / LOGGING,Describes the policy that generated/logged the...
1,case_id,True,True,False,False,False,IDENTIFIER,Join/group/split identifier. Preserved for dat...
2,subscription_id,True,True,False,False,False,IDENTIFIER,Join/group/split identifier. Preserved for dat...
3,customer_id,True,True,False,False,False,IDENTIFIER,Join/group/split identifier. Preserved for dat...
4,decision_seq,False,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...
5,decision_ts_h,False,True,False,False,False,TEMPORAL SOURCE FIELD,Used to derive decision-time temporal features...
6,failure_class_at_decision,False,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...
7,first_failure_class,False,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...
8,rail,True,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...
9,bank_name,True,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...


## 14. Cross-Table Behavioural Audit

For every table, record evidence about:
- scale and cardinality
- missingness
- major distributions
- temporal behaviour
- joins/relationships
- behavioural heterogeneity

The final report must contain only findings actually observed in the data.
Do not copy simulator assumptions as measured facts.

In [15]:
# Compact table-level summary for the final report.
summary_rows = []

for name, df in data.items():
    summary_rows.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    })

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(
    REPORT_DIR / "table_eda_summary.csv",
    index=False,
)


,table,rows,columns,missing_cells,duplicate_rows
0,customers,20000,5,0,0
1,subscriptions,22078,12,0,0
2,payment_attempts,264936,11,238138,0
3,recovery_cases,103026,19,68826,0
4,recovery_events,986365,9,0,0
5,action_log,157423,20,203028,0
6,case_outcomes,103026,17,68826,0
7,ml_datamart,157423,45,203028,0


## 15. Data-Driven Findings and Decision Gate

Write conclusions only after reviewing the actual calculations above.

For each important finding record:
- dataset/table
- variable(s)
- observed statistic
- interpretation
- implication for C3

C2 should finish by deciding:
1. which candidate features have sufficient support/variation;
2. which need transformation;
3. which should be excluded;
4. which require semantic review;
5. whether action support is sufficient for later intervention modelling.

Then move to **C3 — Feature Engineering + Final Feature Contract**.